# BDA POC — Long, mixed-content PDF (text + tables + images)
### SageMaker · Standard output with element extraction + splitter

Companion to the first BDA POC. This one is tuned for a **single long PDF that mixes body text, embedded figures/images, and tables** — e.g. a research report, prospectus, annual report, or a multi-doc application packet.

**What you get out the other end:**
- Full document **text** (per-page markdown + a generative document summary)
- Every **table** as a `pandas.DataFrame` (from BDA's CSV/HTML table representation)
- Every **figure/embedded image** downloaded as an actual `.png` crop, with its generated caption
- Page-level **provenance** (page index + bounding box) on each element — ready for RAG citation

Two non-negotiable settings drive all of this: the **document splitter** (for length) and **`ELEMENT` granularity** (for tables/figures). Both are configured below.


## 0 · Limits that shape the design (current, async API)

| Constraint | Value | Consequence |
|---|---|---|
| Max pages / document (no splitter) | **20** | A long PDF *must* use the splitter |
| Max pages / document (**splitter ENABLED**) | **3,000** | This is your real ceiling |
| Max file size (API) | **500 MB** (200 MB via console) | |
| Formats | PDF, TIFF, JPEG, PNG, DOCX | DOCX is converted to PDF (loses page mapping) |
| Figure captioning | 20 images / page (async) | |
| Languages | EN, DE, ES, FR, IT, PT | No vertical CJK text |
| **Custom blueprint** page cap | **~20 pages / sub-document**, 100 fields max | Custom extraction is *per split sub-doc*, not whole-file |

**Takeaway:** standard output scales to 3,000 pages with the splitter. Custom-field extraction does not scale the same way — it runs per split sub-document. For a long continuous report, lean on **standard output element extraction** (below). For a long *packet* of distinct docs, use the splitter to route each sub-doc to a blueprint (shown in the optional section).


In [ ]:
# Install/upgrade every library this notebook needs, INTO THE CURRENT KERNEL.
%pip install --no-warn-conflicts "boto3>=1.37.6" pandas openpyxl lxml -Uqq
# boto3>=1.37.6 : older boto3 has no 'bedrock-data-automation' service and will raise UnknownServiceError
# pandas        : turn extracted tables into DataFrames
# openpyxl      : lets pandas WRITE tables to an .xlsx workbook
# lxml          : the parser pandas uses for the read_html() table fallback
# --no-warn-conflicts / -U / -q -q : upgrade quietly, no noisy dependency warnings
#
# IMPORTANT: on a fresh kernel, run this cell, then Kernel -> Restart Kernel,
# then continue from the next cell. (If boto3 was already imported at an old
# version, the new client won't be picked up until a restart.)

In [ ]:
import boto3, json, os, io, time     # AWS SDK; JSON parsing; env/paths; in-memory buffers; sleep() for polling
from urllib.parse import urlparse        # to split "s3://bucket/key" strings into bucket + key
from pathlib import Path                  # safe file-path checks (does the PDF exist?)
import pandas as pd                       # tabular handling for the tables we extract

REGION = os.environ.get("AWS_REGION") or "ap-south-1"   # use the kernel's region if set, else force Mumbai (BDA is available there, closest to India)
session     = boto3.Session(region_name=REGION)         # one region-pinned session so every client below is consistent
sts         = session.client("sts")                     # STS: only to look up your account ID
s3_client   = session.client("s3")                      # S3: upload the PDF, read back BDA's JSON + image outputs
bda_client  = session.client("bedrock-data-automation") # BUILD-TIME client: create/update blueprints & projects
bda_runtime = session.client("bedrock-data-automation-runtime")  # RUNTIME client: run jobs + poll status
account_id  = sts.get_caller_identity()["Account"]      # your 12-digit account ID, needed to build the profile ARN below

# Resolve a REAL bucket WITHOUT the SageMaker SDK (avoids the import / no-attribute-Session issue entirely).
BUCKET = os.environ.get("BDA_BUCKET") or f"sagemaker-{REGION}-{account_id}"  # env override, else the conventional default bucket name

def ensure_bucket(name, region):                         # create the bucket if it doesn't already exist
    try:
        s3_client.head_bucket(Bucket=name); return       # exists + accessible -> nothing to do
    except Exception:
        pass                                             # 404 (missing) or 403 -> attempt to create it below
    kwargs = {"Bucket": name}                             # base create args
    if region != "us-east-1":                            # every region EXCEPT us-east-1 REQUIRES an explicit location constraint...
        kwargs["CreateBucketConfiguration"] = {"LocationConstraint": region}  # ...so the bucket lands in ap-south-1, same region as the BDA job
    s3_client.create_bucket(**kwargs); print("Created bucket", name)  # if this raises AccessDenied, create the bucket by hand in the S3 console (region ap-south-1)

ensure_bucket(BUCKET, REGION)                            # guarantee BUCKET exists BEFORE we try to upload the PDF
S3_INPUT, S3_OUTPUT = f"s3://{BUCKET}/bda/input", f"s3://{BUCKET}/bda/output"  # input/output prefixes on the real bucket

def cris_prefix(r):                                      # BDA needs a cross-region-inference (CRIS) profile; its id depends on geography
    if r.startswith("us-") or r.startswith("us_gov"): return "us"    # US regions -> "us"
    if r.startswith("eu-"): return "eu"                  # EU regions -> "eu"
    if r.startswith("ap-"): return "apac"                # Asia-Pacific incl. ap-south-1 (Mumbai) -> "apac"
    raise ValueError(f"Unknown geography for {r}")       # fail loudly instead of guessing a wrong profile
PROFILE_ARN = f"arn:aws:bedrock:{REGION}:{account_id}:data-automation-profile/{cris_prefix(REGION)}.data-automation-v1"  # the MANDATORY profile ARN every invoke requires

print("Region:", REGION, "| Bucket:", BUCKET, "| Profile:", PROFILE_ARN)  # sanity-check the wiring before doing real work

## 1 · Helpers (self-contained)

In [ ]:
def split_s3(uri):                                       # helper: "s3://bucket/key" -> ("bucket", "key")
    p = urlparse(uri); return p.netloc, p.path.lstrip("/")  # netloc = bucket, path = key (drop the leading "/")

def read_s3_json(uri):                                   # read an S3 object and parse it as JSON (BDA results are JSON)
    b, k = split_s3(uri); return json.loads(s3_client.get_object(Bucket=b, Key=k)["Body"].read())  # bytes -> dict

def read_s3_bytes(uri):                                  # read an S3 object as raw bytes (used for figure image crops)
    b, k = split_s3(uri); return s3_client.get_object(Bucket=b, Key=k)["Body"].read()  # return the raw file content

def upload(local, s3_uri):                               # upload a local file to a target S3 URI
    b, k = split_s3(s3_uri); s3_client.upload_file(local, b, k); return s3_uri  # push it, return the URI for convenience

def wait_for_invocation(arn, delay=20, max_iter=60):     # poll a running job until done (async API = you MUST poll)
    for _ in range(max_iter):                            # cap the polls so we never loop forever
        r = bda_runtime.get_data_automation_status(invocationArn=arn); st = r["status"]  # ask BDA for current status
        if st == "Success": print("  ->", st); return r # finished OK -> return the full status response
        if st in ("ClientError", "ServiceError"):        # terminal failure states...
            raise RuntimeError(f"{st}: {r.get('error_type')} / {r.get('error_message')}")  # ...raise instead of hanging
        print("  ...", st); time.sleep(delay)            # still Created/InProgress -> wait, then poll again
    raise TimeoutError("job did not complete")           # ran out of polls -> big jobs may need larger max_iter/delay

def invoke(input_s3, project_arn, stage="LIVE"):         # start a job against a project and block until it finishes
    arn = bda_runtime.invoke_data_automation_async(      # kick off the ASYNC job...
        inputConfiguration={"s3Uri": input_s3},          # the file to process (must be in S3)
        outputConfiguration={"s3Uri": S3_OUTPUT},        # where BDA writes the results
        dataAutomationProfileArn=PROFILE_ARN,            # MANDATORY cross-region inference profile
        dataAutomationConfiguration={"dataAutomationProjectArn": project_arn, "stage": stage},  # which project/config to use
    )["invocationArn"]                                   # pull the job handle out of the response
    print("  invocation:", arn.split("/")[-1]); return wait_for_invocation(arn)  # log short id, then wait for completion

def all_segments(status_response):                       # the splitter can turn ONE pdf into MANY sub-documents ("segments")
    """Splitter produces MANY segments. Return list of (standard_json, custom_json_or_None)."""
    meta = read_s3_json(status_response["outputConfiguration"]["s3Uri"])  # job_metadata.json = the index of all output files
    segs = []                                            # one entry per segment
    for asset in meta.get("output_metadata", []):        # each input asset (we send one file)...
        for seg in asset.get("segment_metadata", []):    # ...but it can have many segments after splitting
            std  = read_s3_json(seg["standard_output_path"]) if seg.get("standard_output_path") else None  # standard result JSON
            cust = (read_s3_json(seg["custom_output_path"])  # custom (blueprint) result...
                    if seg.get("custom_output_status") == "MATCH" and seg.get("custom_output_path") else None)  # ...only if a blueprint matched
            segs.append((std, cust))                      # keep both for this segment
    print(f"  {len(segs)} segment(s) returned by the splitter")  # how many pieces the PDF was split into
    return segs

## 2 · Standard output config — every toggle explained

Each switch below maps directly to something you'll want from a mixed-content doc:

| Setting | Gives you |
|---|---|
| `granularity: ELEMENT` | **tables & figures as discrete elements** (the whole point) — plus DOCUMENT/PAGE/LINE/WORD levels |
| `outputFormat.textFormat: [MARKDOWN, HTML, CSV, PLAIN_TEXT]` | tables arrive with `representation.csv` / `.html` → straight into pandas |
| `additionalFileFormat: ENABLED` | BDA writes **figure crop images** and **per-table CSV files** into your output bucket |
| `generativeField: ENABLED` | document summary, **table summaries**, and **figure captions** |
| `boundingBox: ENABLED` | page index + box per element → RAG provenance / overlays |
| `overrideConfiguration.splitter: ENABLED` | unlocks the **3,000-page** ceiling |


In [ ]:
standard_output_config = {                               # tells BDA WHAT to extract and in WHICH formats
    "document": {                                        # document-modality settings (this notebook is PDF-focused)
        "extraction": {                                  # what structural detail to pull out
            "granularity": {"types": ["DOCUMENT", "PAGE", "ELEMENT", "LINE", "WORD"]},  # ELEMENT = the key one -> tables & figures as objects; the rest add whole-doc / per-page / per-line / per-word views
            "boundingBox": {"state": "ENABLED"}          # attach page + box coordinates to each element (for provenance / citations / overlays)
        },
        "generativeField": {"state": "ENABLED"},         # let the model generate summaries: document summary, table summaries, figure captions
        "outputFormat": {                                # how extracted text/tables are represented
            "textFormat": {"types": ["PLAIN_TEXT", "MARKDOWN", "HTML", "CSV"]},  # CSV + HTML are what make tables loadable into pandas; MARKDOWN gives readable body text
            "additionalFileFormat": {"state": "ENABLED"} # ALSO write side files to S3: cropped figure images + per-table CSV files
        }
    }
}
override_configuration = {"document": {"splitter": {"state": "ENABLED"}}}  # THE setting for long PDFs: splits files >20 pages so BDA can process up to 3000 pages

In [ ]:
PROJECT_NAME = "poc-long-pdf-standard"                   # a stable name so re-runs REUSE the project instead of duplicating it
existing = next((p for p in bda_client.list_data_automation_projects(projectStageFilter="LIVE").get("projects", [])  # look through existing LIVE projects...
                 if p["projectName"] == PROJECT_NAME), None)  # ...for one with this exact name; None if not found
if existing:                                             # already exists -> update it in place (makes this cell idempotent)
    proj = bda_client.update_data_automation_project(
        projectArn=existing["projectArn"],               # target the existing project by ARN
        standardOutputConfiguration=standard_output_config,  # apply our extraction config
        overrideConfiguration=override_configuration)    # keep the splitter enabled
else:                                                    # first run -> create the project
    proj = bda_client.create_data_automation_project(
        projectName=PROJECT_NAME, projectStage="LIVE",   # name + stage (LIVE = usable right away)
        projectDescription="Long mixed-content PDF: text + tables + figures, splitter on",  # human-readable note
        standardOutputConfiguration=standard_output_config,  # same extraction config
        overrideConfiguration=override_configuration)    # splitter on
project_arn = proj["projectArn"]                         # the ARN we pass to every invoke() call
print("Project ARN:", project_arn)                       # confirm which project we'll use

## 3 · Point at your long PDF and invoke

In [ ]:
DOC_LOCAL = "data/long_document.pdf"                     # local path where you drop your PDF (in Studio: drag it into the data/ folder)
DOC_S3    = None                                         # OR set this to an existing "s3://..." URI to skip the upload entirely

os.makedirs("data", exist_ok=True)                       # ensure the ./data folder exists (no error if it already does)
if DOC_S3 is None:                                       # no S3 URI given -> use the local file
    assert Path(DOC_LOCAL).exists(), f"Put a PDF at {DOC_LOCAL} (or set DOC_S3)"  # fail early + clearly if the file is missing
    DOC_S3 = upload(DOC_LOCAL, f"{S3_INPUT}/{Path(DOC_LOCAL).name}")  # upload to S3 (BDA reads only from S3, never local disk)
print("Input:", DOC_S3)                                  # show the S3 location BDA will read from

status = invoke(DOC_S3, project_arn)                     # run the job and block until it succeeds
segments = all_segments(status)                          # read every segment's results back out of S3
standard_outputs = [std for std, _ in segments if std]   # keep just the standard-output JSONs (ignore the custom slot here)

## 4 · Text — assemble the full document

With the splitter on, the doc may come back as several segments. Concatenate page markdown across all of them, and grab the generative summaries.

In [ ]:
import os                                                # path / directory handling
from IPython.display import display, Markdown            # rich rendering inside the notebook (not raw print)

OUT_DIR = "data/output"                                  # EVERYTHING we extract is saved under data/ as requested
os.makedirs(OUT_DIR, exist_ok=True)                      # create data/output if it doesn't exist yet

full_markdown, doc_summaries = [], []                    # collectors: all page markdown, and each segment's doc summary
for std in standard_outputs:                             # iterate every segment (splitter may have produced several)
    doc_summaries.append((std.get("document", {}) or {}).get("summary"))  # the generative whole-document summary, if present
    for page in std.get("pages", []):                    # each page inside this segment
        md_txt = (page.get("representation", {}) or {}).get("markdown", "")  # that page's text as markdown
        full_markdown.append(md_txt)                     # add it to the running list

full_text    = "\n\n".join(m for m in full_markdown if m)   # stitch all non-empty pages into one big markdown string
summary_text = "\n\n".join(s for s in doc_summaries if s)   # combine per-segment summaries into one block

# ---- pretty render in the notebook (formatted Markdown, not print) ----
display(Markdown(f"### Document summary\n\n{summary_text or '_none_'}"))                  # AI summary, formatted
display(Markdown(f"**Pages:** {len(full_markdown)} &nbsp;|&nbsp; **Characters:** {len(full_text):,}"))  # quick stats, bold
display(Markdown("### Body preview (first 1,200 chars)\n\n" + (full_text[:1200] or "_empty_")))  # readable body preview

# ---- save to the data folder ----
open(f"{OUT_DIR}/full_text.md", "w").write(full_text)               # the entire extracted body text
open(f"{OUT_DIR}/document_summary.md", "w").write(summary_text)     # the generative summary
print("saved ->", f"{OUT_DIR}/full_text.md", "|", f"{OUT_DIR}/document_summary.md")  # confirm what was written

## 5 · Tables → pandas DataFrames

Every `TABLE` element carries `representation.csv` (and `.html`). Parse to DataFrames; fall back to HTML for ragged/merged-cell tables. Optionally dump them all to one Excel workbook.

In [ ]:
import os                                                # directory handling
from IPython.display import display, Markdown            # display(df) renders a real HTML table (pretty)

OUT_DIR, TBL_DIR = "data/output", "data/output/tables"   # tables are saved under data/output/tables
os.makedirs(TBL_DIR, exist_ok=True)                      # ensure the folder exists

def collect_tables(standard_outputs):                    # pull every TABLE element out of every segment
    tables = []                                          # each item: segment, page, title, summary, DataFrame
    for seg_i, std in enumerate(standard_outputs):       # seg_i = which sub-document this came from
        for el in std.get("elements", []):               # scan all extracted elements
            if el.get("type") != "TABLE":                # keep only tables
                continue
            rep = el.get("representation", {}) or {}      # the table's representations (csv / html / markdown)
            df = None                                    # will hold the parsed DataFrame
            if rep.get("csv"):                           # preferred: inline CSV string...
                try: df = pd.read_csv(io.StringIO(rep["csv"]))       # ...parse straight into a DataFrame
                except Exception: df = None              # messy? fall through to next method
            if df is None and rep.get("html"):           # fallback: HTML representation...
                try: df = pd.read_html(io.StringIO(rep["html"]))[0]  # ...pandas parses HTML tables (handles merged cells)
                except Exception: df = None
            if df is None and el.get("csv_s3_uri"):       # last resort: BDA wrote a CSV to S3 (additionalFileFormat)
                try: df = pd.read_csv(io.BytesIO(read_s3_bytes(el["csv_s3_uri"])))  # download + parse
                except Exception: df = None
            loc = (el.get("locations") or [{}])[0]        # where the table sits
            tables.append({"segment": seg_i,
                           "page": loc.get("page_index", el.get("page_indices", [None])[0]),  # page number
                           "title": el.get("title"), "summary": el.get("summary"), "df": df})
    return tables

tables = collect_tables(standard_outputs)                # run it
display(Markdown(f"## Extracted tables — {len(tables)} found"))  # pretty section heading

for i, t in enumerate(tables):                           # render + save each table
    title = t["title"] or f"table {i}"                   # fall back to an index if untitled
    cap = f"**p{t['page']} · {title}**" + (f"  \n_{t['summary']}_" if t["summary"] else "")  # caption line
    display(Markdown(cap))                                # show the caption
    if t["df"] is not None:                              # if we parsed it...
        display(t["df"])                                 # ...render as a rich HTML table (pretty)
        t["df"].to_csv(f"{TBL_DIR}/table_{i:02d}_p{t['page']}.csv", index=False)  # save this table as CSV under data/
    else:
        display(Markdown("_(could not parse this table)_"))  # note unparseable tables

if any(t["df"] is not None for t in tables):             # one combined workbook (one sheet per table) under data/
    with pd.ExcelWriter(f"{OUT_DIR}/extracted_tables.xlsx", engine="openpyxl") as xl:  # openpyxl writes .xlsx
        for i, t in enumerate(tables):
            if t["df"] is not None:
                t["df"].to_excel(xl, sheet_name=f"p{t['page']}_t{i}"[:31], index=False)  # Excel caps sheet names at 31 chars
    n = sum(t["df"] is not None for t in tables)
    print("saved ->", f"{OUT_DIR}/extracted_tables.xlsx", f"| {n} CSV(s) in {TBL_DIR}/")

## 6 · Figures / embedded images → downloaded crops

`FIGURE` elements carry `crop_images` (S3 URIs of the cropped image) and a generative `summary` (caption). Download each crop locally.

In [ ]:
import os, csv                                            # directory handling + writing the captions index
from IPython.display import display, Markdown, Image      # inline image rendering

OUT_DIR, FIG_DIR = "data/output", "data/output/figures"   # figures are saved under data/output/figures
os.makedirs(FIG_DIR, exist_ok=True)                      # ensure the folder exists

figures = []                                             # metadata for each downloaded figure
for seg_i, std in enumerate(standard_outputs):           # every segment...
    for el in std.get("elements", []):                   # every element...
        if el.get("type") != "FIGURE":                   # keep only figures / embedded images
            continue
        loc = (el.get("locations") or [{}])[0]           # the figure's location
        page = loc.get("page_index", el.get("page_indices", [None])[0])  # the page it appears on
        for j, crop_uri in enumerate(el.get("crop_images", []) or []):   # each cropped-image S3 URI
            local = f"{FIG_DIR}/seg{seg_i}_p{page}_fig{j}.png"  # save UNDER data/output/figures
            try:
                with open(local, "wb") as f: f.write(read_s3_bytes(crop_uri))  # download + save the crop as PNG
                figures.append({"page": page, "path": local, "caption": el.get("summary") or ""})  # record it
            except Exception as e:
                print("  skip", crop_uri, e)             # one bad image shouldn't abort the loop

display(Markdown(f"## Extracted figures — {len(figures)} found"))  # pretty heading
for fig in figures:                                      # show each image with its caption
    display(Markdown(f"**p{fig['page']}** — {fig['caption'] or '_(no caption)_'}"))  # caption above the image
    display(Image(filename=fig["path"], width=360))      # render the actual crop inline

with open(f"{FIG_DIR}/captions.csv", "w", newline="") as f:  # save a captions index under data/
    w = csv.writer(f); w.writerow(["page", "path", "caption"])  # header row
    for fig in figures: w.writerow([fig["page"], fig["path"], fig["caption"]])  # one row per figure
print("saved ->", len(figures), "image(s) +", f"{FIG_DIR}/captions.csv")

## 7 · Custom blueprint — medical-claim fields (ACTIVE on this PDF)

This section ships a ready-to-run **CMS-1500-style medical-claim blueprint** and runs it on the same `claims-pack.pdf`. It creates the blueprint, attaches it to the splitter project (standard output stays on), re-runs the job, and routes each sub-document to the blueprint.

You now get **both paths on one file**: the standard text / tables / figures from the sections above, **plus** named claim fields per segment (claim number, patient, insured, provider, NPI, diagnosis/CPT codes, charges, service lines).

Blueprint limits to keep in mind: **≤100 leaf fields, ≤30 list fields, ~20 pages per sub-document**. `MATCH` means a blueprint fit that sub-document; `NO_MATCH` is expected for pages in the packet that aren't claim forms.

In [ ]:
import os, json                                           # dirs + JSON handling
from IPython.display import display, Markdown

OUT_DIR = "data/output"; os.makedirs(OUT_DIR, exist_ok=True)

# ---- 1. A ready-to-run medical-claim blueprint (CMS-1500 style) ----
claim_schema = {
    "$schema": "http://json-schema.org/draft-07/schema#",           # JSON-Schema dialect BDA expects
    "description": "US health insurance claim (CMS-1500 style): patient, insured, provider, billing.",  # what this blueprint targets
    "class": "HealthInsuranceClaim",                                 # a label for the document class
    "type": "object",
    "properties": {                                                  # each field = one thing to extract; explicit=verbatim, inferred=derived
        "claim_number":    {"type": "string", "inferenceType": "explicit", "instruction": "Claim or patient control number"},
        "patient_name":    {"type": "string", "inferenceType": "explicit", "instruction": "Full name of the patient"},
        "patient_dob":     {"type": "string", "inferenceType": "explicit", "instruction": "Patient date of birth, YYYY-MM-DD"},
        "insured_name":    {"type": "string", "inferenceType": "explicit", "instruction": "Name of the insured / policy holder"},
        "insured_id":      {"type": "string", "inferenceType": "explicit", "instruction": "Insured member / policy ID number"},
        "insurance_payer": {"type": "string", "inferenceType": "inferred", "instruction": "Name of the insurance company / payer"},
        "provider_name":   {"type": "string", "inferenceType": "explicit", "instruction": "Billing provider or facility name"},
        "provider_npi":    {"type": "string", "inferenceType": "explicit", "instruction": "Provider NPI number"},
        "date_of_service": {"type": "string", "inferenceType": "explicit", "instruction": "Date(s) of service, YYYY-MM-DD"},
        "diagnosis_codes": {"type": "string", "inferenceType": "explicit", "instruction": "ICD-10 diagnosis codes, comma-separated"},
        "total_charge":    {"type": "number", "inferenceType": "explicit", "instruction": "Total charge / billed amount"},
        "amount_paid":     {"type": "number", "inferenceType": "explicit", "instruction": "Amount already paid, if shown"},
        "balance_due":     {"type": "number", "inferenceType": "inferred", "instruction": "Outstanding balance (total_charge minus amount_paid)"}
    },
    "definitions": {                                                 # reusable shape for the service-line list below
        "ServiceLine": {
            "type": "object",
            "properties": {
                "date_of_service": {"type": "string", "inferenceType": "explicit", "instruction": "Service line date"},
                "cpt_code":        {"type": "string", "inferenceType": "explicit", "instruction": "CPT / HCPCS procedure code"},
                "description":     {"type": "string", "inferenceType": "explicit", "instruction": "Procedure description"},
                "charge":          {"type": "number", "inferenceType": "explicit", "instruction": "Charge for this line"}
            }
        }
    },
    "service_lines": {"type": "array", "instruction": "Every procedure/service line on the claim",  # the list field
                      "items": {"$ref": "#/definitions/ServiceLine"}}
}

# ---- 2. Create the blueprint (idempotent by name so re-runs don't duplicate) ----
BP_NAME = "poc-health-claim"                                         # stable blueprint name
_existing = next((b for b in bda_client.list_blueprints(blueprintStageFilter="ALL").get("blueprints", [])  # search existing...
                  if b.get("blueprintName") == BP_NAME), None)       # ...by name
if _existing:                                                       # already exists -> update its schema in place
    CUSTOM_BLUEPRINT_ARN = bda_client.update_blueprint(
        blueprintArn=_existing["blueprintArn"], blueprintStage="LIVE",
        schema=json.dumps(claim_schema))["blueprint"]["blueprintArn"]
else:                                                              # first run -> create it
    CUSTOM_BLUEPRINT_ARN = bda_client.create_blueprint(
        blueprintName=BP_NAME, type="DOCUMENT", blueprintStage="LIVE",
        schema=json.dumps(claim_schema))["blueprint"]["blueprintArn"]
print("Blueprint ARN:", CUSTOM_BLUEPRINT_ARN)                       # the ARN we attach next

# ---- 3. Attach the blueprint to the SAME splitter project (standard + custom together) ----
bda_client.update_data_automation_project(
    projectArn=project_arn,
    standardOutputConfiguration=standard_output_config,             # keep standard extraction ON...
    customOutputConfiguration={"blueprints": [                       # ...and ADD custom output
        {"blueprintArn": CUSTOM_BLUEPRINT_ARN, "blueprintStage": "LIVE"}]},
    overrideConfiguration=override_configuration)                   # splitter STAYS ON -> each sub-doc gets routed to the blueprint
print("Attached blueprint to project; re-running the job...")

# ---- 4. Re-run; now every segment carries BOTH standard and custom output ----
status2   = invoke(DOC_S3, project_arn)                             # invoke again, routing enabled
segments2 = all_segments(status2)                                  # read all segments back out of S3

# ---- 5. Pretty-display + save the custom fields per matched segment ----
matched = sum(1 for _, c in segments2 if c)                        # how many sub-docs the blueprint matched
display(Markdown(f"## Custom blueprint output — {matched}/{len(segments2)} segment(s) matched"))
custom_results = []                                                # collect for saving to disk
for seg_i, (std, cust) in enumerate(segments2):                    # walk every sub-document
    fields = (cust or {}).get("inference_result") or cust          # the extracted fields dict (present only if matched)
    display(Markdown(f"**Segment {seg_i} — {'MATCH' if cust else 'NO_MATCH'}**"))  # routing status per sub-doc
    if fields:                                                     # matched -> render fields as a neat 2-column table
        scalars = {k: v for k, v in fields.items() if not isinstance(v, (list, dict))}  # simple fields only
        display(pd.DataFrame(scalars.items(), columns=["field", "value"]))              # pretty key/value table
        lines = fields.get("service_lines")                        # the repeating service-line list
        if isinstance(lines, list) and lines:
            display(Markdown("_service lines:_")); display(pd.DataFrame(lines))          # render the line items as a table
        custom_results.append({"segment": seg_i, "fields": fields})  # keep for JSON export
    else:
        display(Markdown("_(no blueprint matched this segment — not a claim form)_"))

with open(f"{OUT_DIR}/custom_fields.json", "w") as f:              # save all custom extractions under data/
    json.dump(custom_results, f, indent=2, ensure_ascii=False, default=str)  # default=str handles Decimals etc.
print("saved ->", f"{OUT_DIR}/custom_fields.json")

## 8 · RAG-ready, element-aware chunking

The reason you extracted elements instead of a blob: chunk **by element**, keep tables whole, and attach figure captions + page provenance. This preserves structure that naive character-splitting destroys — tables stay queryable, figures stay findable, and every chunk carries a page cite.

In [ ]:
import os, json                                           # directory + JSON handling
from IPython.display import display, Markdown

OUT_DIR = "data/output"; os.makedirs(OUT_DIR, exist_ok=True)

def build_chunks(standard_outputs):                      # turn elements into RAG-ready chunks that carry provenance
    chunks = []
    for seg_i, std in enumerate(standard_outputs):       # every segment
        for el in std.get("elements", []):               # every element
            loc = (el.get("locations") or [{}])[0]        # element location
            page = loc.get("page_index", el.get("page_indices", [None])[0])  # page number, kept for citation
            rep = el.get("representation", {}) or {}      # element's representations
            etype = el.get("type")                        # TEXT / TABLE / FIGURE
            if etype == "TABLE":                          # tables: keep the WHOLE table + summary in one chunk
                content = rep.get("markdown") or rep.get("html") or rep.get("csv") or ""
                content = f"[TABLE] {el.get('title') or ''}\n{content}\nSummary: {el.get('summary') or ''}"
            elif etype == "FIGURE":                       # figures: index the caption text
                content = f"[FIGURE] {el.get('summary') or '(image)'}"
            else:                                         # plain text elements
                content = rep.get("markdown") or rep.get("text") or ""
            if content.strip():                           # skip empty elements
                chunks.append({"type": etype, "page": page, "segment": seg_i,
                               "reading_order": el.get("reading_order"), "content": content})
    return chunks

chunks = build_chunks(standard_outputs)                  # build them

counts = {"TEXT": 0, "TABLE": 0, "FIGURE": 0}            # tally chunk types for a pretty summary
for c in chunks: counts[c["type"]] = counts.get(c["type"], 0) + 1
display(Markdown(f"## RAG chunks — {len(chunks)} total"))
display(pd.DataFrame([counts]))                          # one-row DataFrame -> renders as a neat table

with open(f"{OUT_DIR}/chunks.json", "w") as f:           # save the chunks as JSON under data/ (ready for a vector store)
    json.dump(chunks, f, indent=2, ensure_ascii=False)
print("saved ->", f"{OUT_DIR}/chunks.json")

## 8b · One consolidated, pretty report (saved to `data/output/report.html`)

Bundles the summary, every table, and every figure (images embedded, so the file is self-contained) into a single HTML report — rendered inline **and** written to `data/output/report.html`. Run this after the text / tables / figures / chunks cells above.

In [ ]:
import os, base64, html                                   # dirs, image embedding, HTML escaping
from IPython.display import display, HTML

OUT_DIR = "data/output"; os.makedirs(OUT_DIR, exist_ok=True)

def _b64(path):                                          # base64-embed an image so the HTML report is self-contained
    with open(path, "rb") as f: return base64.b64encode(f.read()).decode()

parts = ["<h1>BDA extraction report</h1>"]               # start building the HTML document
parts.append(f"<p><b>Segments:</b> {len(standard_outputs)} &nbsp; <b>Tables:</b> {len(tables)} "
             f"&nbsp; <b>Figures:</b> {len(figures)} &nbsp; <b>Chunks:</b> {len(chunks)}</p>")  # headline stats
parts.append("<h2>Summary</h2><p>" + html.escape(summary_text or "(none)").replace(chr(10), "<br>") + "</p>")  # doc summary

parts.append("<h2>Tables</h2>")                          # render each parsed table as HTML
for i, t in enumerate(tables):
    parts.append(f"<h4>p{t['page']} &middot; {html.escape(t['title'] or f'table {i}')}</h4>")
    parts.append(t["df"].to_html(index=False) if t["df"] is not None else "<i>could not parse</i>")

parts.append("<h2>Figures</h2>")                         # embed each crop + caption
for fig in figures:
    parts.append(f"<div style='margin:10px 0'><img src='data:image/png;base64,{_b64(fig['path'])}' "
                 f"width='320'><br><small>p{fig['page']} — {html.escape(fig['caption'] or '')}</small></div>")

report_html = "<div style='font-family:system-ui,sans-serif;line-height:1.4'>" + "".join(parts) + "</div>"  # wrap
open(f"{OUT_DIR}/report.html", "w").write(report_html)   # SAVE the consolidated report under data/
display(HTML(report_html))                               # and render it right here, pretty
print("saved ->", f"{OUT_DIR}/report.html")

## 9 · Cleanup (optional)

In [ ]:
# Deleting the project is OPTIONAL. Uncomment to remove it when you're done.
# bda_client.delete_data_automation_project(projectArn=project_arn)  # removes the project (any blueprints are separate resources)
# print("deleted project")

---
### Notes for your pipeline
- **Splitter is the whole ballgame for length.** Forget it and you're capped at 20 pages. It also means *always* loop over segments — never assume one result.
- **Standard output does the structural heavy lifting** (text + tables + figures). Reach for custom blueprints only when you need *specific named fields*, and remember they run per ≤20-page sub-doc.
- **`additionalFileFormat: ENABLED` costs S3 writes** (crop images + table CSVs land in your output bucket). If you only need inline `representation.csv`, you can leave it off and save the storage.
- **Generative fields cost latency.** If you don't need summaries/captions, disabling `generativeField` measurably speeds up big jobs.
- **Element-aware chunking >> character chunking** for RAG over reports — tables and figures survive as first-class, citable units.
- **ScoreNLearn fit:** an admissions "packet PDF" (transcript + scores + SOP + financials) is the packet-routing case (section 7); a single long university brochure or policy doc is the standard-element case (sections 4–6).
